# DATA 255 Lab 1 - ImageNet20 Classification (Working Version)

Based on proven working approach. Train CNN from scratch, export to ONNX.

**Requirements:**
- Input: 240×240×3 RGB images
- Output: 20 classes
- No pretrained weights
- Target: 60%+ validation accuracy

In [ ]:
# Cell 1: Mount Drive & Install packages
from google.colab import drive
drive.mount('/content/drive')

# Install all required packages
!pip install -q onnx onnxruntime onnxscript

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Cell 2: Imports and Config
import os
import csv
import zipfile
from dataclasses import dataclass
from typing import Tuple, List

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from tqdm import tqdm

@dataclass
class Config:
    # === MODIFY THIS PATH TO YOUR DRIVE FOLDER ===
    drive_data_path: str = "/content/drive/MyDrive/DL Lab1"

    # Extracted paths (auto-configured)
    train_images_root: str = "/content/imagenet_train20/imagenet_train20a"
    val_images_root: str = "/content/imagenet_val20/imagenet_val20"
    train_list_file: str = ""  # Will be set from drive_data_path
    val_list_file: str = ""    # Will be set from drive_data_path

    # Model settings
    num_classes: int = 20
    input_size: int = 240

    # Training settings
    batch_size: int = 64
    num_epochs: int = 40
    learning_rate: float = 1.5e-3
    weight_decay: float = 1e-4
    label_smoothing: float = 0.1

    # Output settings
    output_dir: str = "/content/outputs"
    best_model_path: str = "best_model.pth"
    log_csv: str = "training_log.csv"
    onnx_path: str = "model_imagenet20.onnx"
    onnx_opset: int = 11

CONFIG = Config()
CONFIG.train_list_file = os.path.join(CONFIG.drive_data_path, "imagenet_train20.txt")
CONFIG.val_list_file = os.path.join(CONFIG.drive_data_path, "imagenet_val20.txt")

os.makedirs(CONFIG.output_dir, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Using device: cuda
GPU: Tesla T4


In [ ]:
# Cell 3: Extract Dataset
print("Extracting training data...")
train_zip = os.path.join(CONFIG.drive_data_path, "imagenet_train20.zip")
with zipfile.ZipFile(train_zip, 'r') as z:
    z.extractall("/content")

print("Extracting validation data...")
val_zip = os.path.join(CONFIG.drive_data_path, "imagenet_val20.zip")
with zipfile.ZipFile(val_zip, 'r') as z:
    z.extractall("/content")

print("Extraction complete!")

# Show what was actually extracted
print("\n=== Contents of /content after extraction ===")
for item in os.listdir("/content"):
    item_path = os.path.join("/content", item)
    if os.path.isdir(item_path) and not item.startswith('.'):
        try:
            count = len(os.listdir(item_path))
            print(f"  {item}/ ({count} items)")
        except:
            print(f"  {item}/")
    elif not item.startswith('.'):
        print(f"  {item}")

Extracting training data...
Extracting validation data...
Extraction complete!

=== Contents of /content after extraction ===
  imagenet_val20/ (1000 items)
  drive/ (4 items)
  imagenet_train20a/ (20 items)
  outputs/ (0 items)
  sample_data/ (6 items)


In [ ]:
# Cell 4: Auto-detect image roots and create Dataset

# Find all directories in /content that might contain our images
print("=== Searching for image directories ===")

def find_all_image_dirs(base_path="/content"):
    """Find ALL directories containing images anywhere under base_path"""
    image_dirs = []
    for root, dirs, files in os.walk(base_path):
        # Skip drive and sample_data folders
        if 'drive' in root or 'sample_data' in root:
            continue
        for f in files:
            if f.lower().endswith(('.jpeg', '.jpg', '.png')):
                if root not in image_dirs:
                    image_dirs.append(root)
                break
    return image_dirs

all_image_dirs = find_all_image_dirs("/content")
print(f"Found {len(all_image_dirs)} directories with images")

for d in all_image_dirs[:5]:
    count = len([f for f in os.listdir(d) if f.lower().endswith(('.jpeg', '.jpg', '.png'))])
    print(f"  {d} ({count} images)")
if len(all_image_dirs) > 5:
    print(f"  ... and {len(all_image_dirs) - 5} more directories")

# Separate train and val based on folder names
train_dirs = [d for d in all_image_dirs if 'train' in d.lower()]
val_dirs = [d for d in all_image_dirs if 'val' in d.lower()]

print(f"\nTrain dirs: {len(train_dirs)}, Val dirs: {len(val_dirs)}")

# Sample file check
if train_dirs:
    sample_files = [f for f in os.listdir(train_dirs[0]) if f.lower().endswith(('.jpeg', '.jpg', '.png'))][:3]
    print(f"Sample train files: {sample_files}")

class ImageNet20Dataset(Dataset):
    def __init__(self, image_dirs: list, list_file: str, transform=None):
        self.transform = transform
        self.samples: List[Tuple[str, int]] = []

        # Build index of ALL images in ALL directories
        print(f"Indexing images from {len(image_dirs)} directories...")
        image_index = {}
        for img_dir in image_dirs:
            if not os.path.exists(img_dir):
                continue
            for item in os.listdir(img_dir):
                if item.lower().endswith(('.jpeg', '.jpg', '.png')):
                    image_index[item] = os.path.join(img_dir, item)

        print(f"Found {len(image_index)} images in index")

        # Parse label file and match with indexed images
        with open(list_file, "r") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                parts = line.split()
                if len(parts) != 2:
                    continue
                rel_path, label_str = parts
                label = int(label_str)

                # Get just the filename
                filename = os.path.basename(rel_path)

                # Look up in index
                if filename in image_index:
                    self.samples.append((image_index[filename], label))

        print(f"Loaded {len(self.samples)} samples from {list_file}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        with Image.open(img_path) as img:
            img = img.convert("RGB")
        if self.transform is not None:
            img = self.transform(img)
        return img, label


def create_dataloaders(cfg: Config, train_dirs, val_dirs):
    train_transform = transforms.Compose([
        transforms.RandomResizedCrop(cfg.input_size, scale=(0.6, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.RandomRotation(15),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    val_transform = transforms.Compose([
        transforms.Resize(cfg.input_size + 16),
        transforms.CenterCrop(cfg.input_size),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    train_dataset = ImageNet20Dataset(
        image_dirs=train_dirs,
        list_file=cfg.train_list_file,
        transform=train_transform,
    )
    val_dataset = ImageNet20Dataset(
        image_dirs=val_dirs,
        list_file=cfg.val_list_file,
        transform=val_transform,
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=cfg.batch_size,
        shuffle=True,
        num_workers=2,
        pin_memory=True,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=cfg.batch_size,
        shuffle=False,
        num_workers=2,
        pin_memory=True,
    )

    return train_loader, val_loader

# Create dataloaders
train_loader, val_loader = create_dataloaders(CONFIG, train_dirs, val_dirs)
print(f"\nTrain batches: {len(train_loader)}, Val batches: {len(val_loader)}")

=== Searching for image directories ===
Found 21 directories with images
  /content/imagenet_val20 (1000 images)
  /content/imagenet_train20a/n03938244 (300 images)
  /content/imagenet_train20a/n03721384 (300 images)
  /content/imagenet_train20a/n03804744 (300 images)
  /content/imagenet_train20a/n03016953 (300 images)
  ... and 16 more directories

Train dirs: 20, Val dirs: 1
Sample train files: ['n03938244_12113.JPEG', 'n03938244_12905.JPEG', 'n03938244_15719.JPEG']
Indexing images from 20 directories...
Found 6000 images in index
Loaded 6000 samples from /content/drive/MyDrive/DL Lab1/imagenet_train20.txt
Indexing images from 1 directories...
Found 1000 images in index
Loaded 1000 samples from /content/drive/MyDrive/DL Lab1/imagenet_val20.txt

Train batches: 94, Val batches: 16


In [ ]:
# Cell 5: Model Architecture (Proven working design)
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, num_convs=1):
        super().__init__()
        layers = []
        layers.append(nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False))
        layers.append(nn.BatchNorm2d(out_channels))
        layers.append(nn.ReLU(inplace=True))

        for _ in range(num_convs - 1):
            layers.append(nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False))
            layers.append(nn.BatchNorm2d(out_channels))
            layers.append(nn.ReLU(inplace=True))

        self.conv = nn.Sequential(*layers)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

    def forward(self, x):
        x = self.conv(x)
        x = self.pool(x)
        return x


class ImageNet20Net(nn.Module):
    def __init__(self, num_classes=20):
        super().__init__()

        self.features = nn.Sequential(
            ConvBlock(3, 32, num_convs=1),      # 240 -> 120
            ConvBlock(32, 64, num_convs=1),     # 120 -> 60
            ConvBlock(64, 128, num_convs=2),    # 60 -> 30
            ConvBlock(128, 256, num_convs=2),   # 30 -> 15
        )

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.5),
            nn.Linear(256, num_classes),
        )

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1.0)
                nn.init.constant_(m.bias, 0.0)
            elif isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0.0)

    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = self.classifier(x)
        return x

# Create model
model = ImageNet20Net(num_classes=CONFIG.num_classes).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total_params:,}")

# Test forward pass
test_input = torch.randn(1, 3, 240, 240).to(device)
test_output = model(test_input)
print(f"Input shape: {test_input.shape}")
print(f"Output shape: {test_output.shape}")

Model parameters: 1,197,876
Input shape: torch.Size([1, 3, 240, 240])
Output shape: torch.Size([1, 20])


In [ ]:
# Cell 6: Training Functions
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    running_correct = 0
    running_total = 0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = outputs.max(1)
        running_correct += (preds == labels).sum().item()
        running_total += labels.size(0)

    epoch_loss = running_loss / running_total
    epoch_acc = running_correct / running_total
    return epoch_loss, epoch_acc


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    running_correct = 0
    running_total = 0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        outputs = model(images)
        loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)
        _, preds = outputs.max(1)
        running_correct += (preds == labels).sum().item()
        running_total += labels.size(0)

    epoch_loss = running_loss / running_total
    epoch_acc = running_correct / running_total
    return epoch_loss, epoch_acc


def save_log_header(csv_path: str):
    if os.path.exists(csv_path):
        return
    with open(csv_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["epoch", "train_loss", "train_acc", "val_loss", "val_acc"])


def append_log_row(csv_path: str, epoch: int, train_loss: float, train_acc: float,
                   val_loss: float, val_acc: float):
    with open(csv_path, "a", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([epoch, train_loss, train_acc, val_loss, val_acc])

In [ ]:
# Cell 7: Train the Model
criterion = nn.CrossEntropyLoss(label_smoothing=CONFIG.label_smoothing)
optimizer = optim.Adam(
    model.parameters(),
    lr=CONFIG.learning_rate,
    weight_decay=CONFIG.weight_decay,
)
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=CONFIG.num_epochs,
    eta_min=1e-6,
)

best_val_acc = 0.0
best_model_path = os.path.join(CONFIG.output_dir, CONFIG.best_model_path)
log_csv_path = os.path.join(CONFIG.output_dir, CONFIG.log_csv)
save_log_header(log_csv_path)

print(f"{'Epoch':>6} | {'Train Loss':>10} | {'Train Acc':>10} | {'Val Loss':>10} | {'Val Acc':>10}")
print("-" * 60)

for epoch in range(1, CONFIG.num_epochs + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)
    scheduler.step()

    print(f"{epoch:>6} | {train_loss:>10.4f} | {train_acc*100:>9.2f}% | {val_loss:>10.4f} | {val_acc*100:>9.2f}%", end="")

    append_log_row(log_csv_path, epoch, train_loss, train_acc, val_loss, val_acc)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), best_model_path)
        print(f" *NEW BEST*")
    else:
        print()

print(f"\nTraining complete!")
print(f"Best Val Acc: {best_val_acc * 100:.2f}%")
print(f"Model saved to: {best_model_path}")

 Epoch | Train Loss |  Train Acc |   Val Loss |    Val Acc
------------------------------------------------------------
     1 |     3.3799 |     10.55% |     2.7442 |     17.10% *NEW BEST*
     2 |     2.7783 |     15.42% |     2.6292 |     22.30% *NEW BEST*
     3 |     2.7146 |     18.73% |     2.5312 |     25.40% *NEW BEST*
     4 |     2.6690 |     19.88% |     2.5740 |     21.50%
     5 |     2.6527 |     20.08% |     2.4668 |     26.30% *NEW BEST*
     6 |     2.5997 |     22.20% |     2.5125 |     25.80%
     7 |     2.5651 |     23.65% |     2.4121 |     28.80% *NEW BEST*
     8 |     2.5124 |     26.02% |     2.3225 |     31.40% *NEW BEST*
     9 |     2.4755 |     27.78% |     2.3365 |     33.50% *NEW BEST*
    10 |     2.4239 |     29.62% |     2.2175 |     35.80% *NEW BEST*
    11 |     2.3992 |     30.62% |     2.3283 |     34.00%
    12 |     2.3469 |     32.60% |     2.2584 |     34.60%
    13 |     2.3330 |     34.08% |     2.1816 |     38.20% *NEW BEST*
    14 |     2

In [ ]:
# Cell 8: Export to ONNX
def export_to_onnx(cfg: Config):
    # Load model on CPU for ONNX export
    cpu_device = torch.device("cpu")
    model = ImageNet20Net(num_classes=cfg.num_classes)
    best_model_path = os.path.join(cfg.output_dir, cfg.best_model_path)
    model.load_state_dict(torch.load(best_model_path, map_location=cpu_device))
    model.to(cpu_device)
    model.eval()

    dummy_input = torch.randn(1, 3, cfg.input_size, cfg.input_size, device=cpu_device)

    onnx_path = os.path.join(cfg.output_dir, cfg.onnx_path)
    print(f"Exporting to ONNX: {onnx_path}")

    # Force legacy exporter (dynamo=False) to avoid PyTorch 2.x producing tiny broken files
    torch.onnx.export(
        model,
        dummy_input,
        onnx_path,
        input_names=["input"],
        output_names=["logits"],
        opset_version=cfg.onnx_opset,
        do_constant_folding=True,
        dynamic_axes=None,
        dynamo=False,
    )

    # Verify file size
    file_size = os.path.getsize(onnx_path) / (1024 * 1024)
    print(f"ONNX export complete!")
    print(f"File size: {file_size:.2f} MB")

    if file_size < 1.0:
        print("WARNING: File is very small - export may have failed!")
    else:
        print("File size looks correct!")

    return onnx_path

onnx_path = export_to_onnx(CONFIG)


Exporting to ONNX: /content/outputs/model_imagenet20.onnx


/tmp/ipython-input-3351/2559098143.py:17: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


ONNX export complete!
File size: 4.57 MB
File size looks correct!


In [ ]:
# Cell 9: Verify ONNX Model
import onnx
import onnxruntime as ort

# Load and check ONNX model
onnx_model = onnx.load(onnx_path)
onnx.checker.check_model(onnx_model)
print("ONNX model is valid!")

# Test inference with ONNX Runtime
session = ort.InferenceSession(onnx_path)
test_input = torch.randn(1, 3, 240, 240).numpy()
outputs = session.run(None, {"input": test_input})
print(f"ONNX inference output shape: {outputs[0].shape}")
print(f"Sample predictions: {outputs[0][0, :5]}")

ONNX model is valid!
ONNX inference output shape: (1, 20)
Sample predictions: [-4.08252   -3.8455212  6.123221  10.006734   2.878673 ]


In [ ]:
# Cell 10: Create Calibration Images
import random

def create_calibration_images(image_dirs):
    """Create calibration_images.zip with ~200 random images"""
    calib_dir = os.path.join(CONFIG.output_dir, "calibration_images")
    os.makedirs(calib_dir, exist_ok=True)

    # Collect all training images from all directories
    all_images = []
    for img_dir in image_dirs:
        if not os.path.exists(img_dir):
            continue
        for f in os.listdir(img_dir):
            if f.lower().endswith(('.jpg', '.jpeg', '.png')):
                all_images.append(os.path.join(img_dir, f))

    print(f"Found {len(all_images)} images for calibration")

    # Sample ~200 images
    num_samples = min(200, len(all_images))
    selected = random.sample(all_images, num_samples)

    # Copy images
    for i, src in enumerate(selected):
        dst = os.path.join(calib_dir, f"calib_{i:04d}.jpg")
        Image.open(src).convert('RGB').resize((240, 240)).save(dst, 'JPEG', quality=95)

    # Create zip
    zip_path = os.path.join(CONFIG.output_dir, "calibration_images.zip")
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for f in os.listdir(calib_dir):
            zf.write(os.path.join(calib_dir, f), f)

    print(f"Created {zip_path} with {num_samples} images")
    return zip_path

# Use the train_dirs from Cell 4
calib_zip = create_calibration_images(train_dirs)

Found 6000 images for calibration
Created /content/outputs/calibration_images.zip with 200 images
